# Basics
This notebook contains examples of basic functionality using ``Source`` and ``Transformation`` objects.

## Making sources and transformations
A source is a data placeholder, and a transformation is a source that has methods for manipulating data. We can create a source as,

In [1]:
from py_online_forecast import *

data_source = Source("data")

Transformations are then used to define dataflows using sources as inputs. A simple ``Transformation`` is the ``One`` transformation, that creates an array of ones of the same size as the input data.

In [2]:
constant = One(data_source)

Once data becomes available, any transformation can be evaluated on it. To do so, we need to map the data value to its source,

In [3]:
import numpy as np

# Create a data value
data_value = np.array([1, 2, 3, 4, 5])

# Apply the transformation
constant({data_source: data_value})


array([1., 1., 1., 1., 1.])

Sources (and thus transformations) also allow for common operators that tacitly create new transformations,

In [4]:
# Define a new transformation by adding 1 to the data source
data_plus_one = data_source + 1

# Create another transformation by indexing the new transformation
first_three = data_plus_one[:3]

# Apply the new transformation
first_three({data_source: data_value})

array([2, 3, 4])

This way, sources and transformations can be composed freely. Be careful that the output values of the transformations match the expected inputs for subsequent transformations.

For convenience, we can use a built-in source ``DEFAULT_SOURCE`` when we do not wish to specify multiple sources. Then the above reduces to,

In [5]:
# Create transform using DEFAULT_SOURCE
data_plus_one = DEFAULT_SOURCE + 1
first_three = data_plus_one[:3]

# Evaluate, without specifying the source.
first_three(data_value) # data_value is automatically mapped to DEFAULT_SOURCE.

array([2, 3, 4])

## Online updates

The ``Transformation`` subclass instances (e.g. ``constant``, ``data_plus_one`` etc.) are intended to hold static parameters and define pure functions on input data. In practice, not all data may be available at once, and we need to apply transformations incrementally. In such cases, transformation may need to carry a state. An example is the ``LowPass`` transformation, which tracks previously filtered values to make new updates.

In [6]:
# Make some data
data = np.random.rand(6, 1)

# Create a low-pass filter transformation
low_pass = LowPass(DEFAULT_SOURCE, alpha=0.1)

# Apply the low-pass filter to the data
first_three = low_pass(data[:3])

last_three = low_pass(data[3:6])

all_data = low_pass(data)

# Check where they are the same
np.isclose(np.concatenate([first_three, last_three]), all_data)

array([[ True],
       [ True],
       [ True],
       [False],
       [False],
       [False]])

The low-pass filter does not track the state unless explicitly asked. As a consequence we get different results, when applying the filter in steps versus in batch. To keep track of the state, we may either return it or ask the transform to keep track of the state.

In [7]:
# Request the recursion parameters (state) used in the low-pass filter
first_three, state = low_pass(data[:3], return_recursion_pars = True)
state

{ToArray: None, LowPass: array([0.74830762])}

Note, the output is a tuple of the output data, and a dict of states. The state dict has keys that are ``Transformation`` objects, and values that are states returned by those transformations. For transformations that are composed of other transformations, the returned state will have multiple entries, one for each transformation.

We can pass the state back to transformation to resume where we left of,

In [8]:
last_three = low_pass(data[3:6], recursion_pars = state)

np.isclose(np.concatenate([first_three, last_three]), all_data)

array([[ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True]])

Transformations can also handle this manually, if the ``track_state`` flag is passed.

In [9]:
first_three = low_pass(data[:3], track_state = True)
last_three = low_pass(data[3:6], track_state = True)
np.isclose(np.concatenate([first_three, last_three]), all_data)

array([[ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True]])

## Note on parameters
Parameters for transformations are passed at instantiation. There is no universal interface to update parameters, but some transformations may allow parameters to be updated, e.g.,

In [10]:
low_pass.alpha = 0.2

print("With alpha = 0.1:")
print(all_data)

print("With alpha = 0.2:")
print(low_pass(data))

With alpha = 0.1:
[[0.09052034]
 [0.50012388]
 [0.74830762]
 [0.23973735]
 [0.91326975]
 [0.58929347]]
With alpha = 0.2:
[[0.09052034]
 [0.45461238]
 [0.71162934]
 [0.2889095 ]
 [0.84826725]
 [0.61229033]]
